# Colab Entry Notebook

This notebook boots up a Google Colab runtime for the `ese645` project. It handles cloning the private repository with a stored GitHub token, installing Python dependencies, and provides helper cells for development and evaluation workflows such as Direct Inversion testing.


## Prerequisites
- Store a personal access token with repo scope in Colab: `Runtime ▸ Run all...` will prompt for `GITHUB_TOKEN` via *Secrets* (Menu ▸ `Tools ▸ Secrets`).
- (Optional) Upload or mount the PIE-Bench dataset to `/content/ese645/data/PIE-Bench_v1`.
- Make sure the Colab runtime has a GPU (`Runtime ▸ Change runtime type ▸ GPU`).


In [ ]:
import os
from pathlib import Path

try:
    from google.colab import userdata  # type: ignore
except ImportError as exc:  # noqa: F401
    raise RuntimeError("This notebook is intended to run in Google Colab.") from exc

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN") or userdata.get("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise ValueError("Please populate a `GITHUB_TOKEN` secret in Colab (Tools ▸ Secrets).")

os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
os.environ.setdefault("GIT_TERMINAL_PROMPT", "0")

REPO_OWNER = "roastedpotato66"
REPO_NAME = "ese645"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
WORKDIR = Path("/content") / REPO_NAME

print(f"Using repository: {REPO_URL}\nWorking directory: {WORKDIR}")


In [ ]:
import subprocess

if WORKDIR.exists():
    print("Repository already present. Pulling latest changes...")
    subprocess.run(["git", "-C", str(WORKDIR), "pull"], check=True)
else:
    print("Cloning private repository via token...")
    clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    subprocess.run(["git", "clone", clone_url, str(WORKDIR)], check=True)
    subprocess.run(["git", "-C", str(WORKDIR), "remote", "set-url", "origin", REPO_URL], check=True)
    print("Clone complete.")


In [ ]:
%cd /content/ese645

%pip install -q -r requirements.txt


In [ ]:
# Quick smoke test (adjust paths / steps as needed) 
!python scripts/test_ddim_single.py --device cuda --num_steps 5


## Benchmark Runs
Use the commands below to generate PIE-Bench outputs and compute metrics. Adjust `--num_steps` or device arguments as needed for the Colab GPU.


In [ ]:
# Run Direct Inversion on a small subset (first 5 images of category 0)
!python scripts/run_pie_bench_sample.py --device cuda --num_steps 5 --num_images 5 --category 0


In [ ]:
# Evaluate generated results and save metrics
!python scripts/run_evaluation.py --tgt_image_folder outputs/direct_inversion/annotation_images --output_csv results/metrics.csv


## Repository Sync (pull only)
The commands below keep the Colab workspace up to date with the remote repository. All development should happen on your local machine; avoid pushing from Colab.


In [ ]:
# Refresh local checkout from origin/phase1 (default branch)
!git -C /content/ese645 fetch origin
!git -C /content/ese645 pull origin phase1
